# 7.1 分类评估指标

> **模块七：模型评估与优化** | NOAI竞赛课程
>
> 本节约 **2课时**，重点掌握分类模型的各类评估指标

---

## 📚 学习目标

1. 理解混淆矩阵的构成（TP, FP, TN, FN）
2. 掌握准确率、精确率、召回率、F1分数的计算与适用场景
3. 理解ROC曲线与AUC值的含义
4. 了解PR曲线及其与ROC曲线的区别
5. 掌握多分类问题的评估方法（macro/micro/weighted平均）
6. 使用sklearn计算各类指标并绘制ROC曲线和PR曲线

---

## 1. 混淆矩阵（Confusion Matrix）

混淆矩阵是分类任务中最基础的评估工具，它以矩阵形式展示模型预测结果与真实标签的对应关系。

### 1.1 二分类混淆矩阵

对于二分类问题（正类 Positive / 负类 Negative），混淆矩阵如下：

| | 预测为正（Predicted Positive） | 预测为负（Predicted Negative） |
|---|---|---|
| **实际为正（Actual Positive）** | **TP**（真正例） | **FN**（假负例） |
| **实际为负（Actual Negative）** | **FP**（假正例） | **TN**（真负例） |

### 1.2 四个核心概念

- **TP（True Positive，真正例）**：实际是正类，预测也是正类 ✅
- **FP（False Positive，假正例）**：实际是负类，预测为正类 ❌（第一类错误）
- **TN（True Negative，真负例）**：实际是负类，预测也是负类 ✅
- **FN（False Negative，假负例）**：实际是正类，预测为负类 ❌（第二类错误）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 生成示例数据
X, y = make_classification(n_samples=1000, n_features=10, n_informative=5,
                          n_classes=2, weights=[0.7, 0.3], random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 训练模型
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# 计算并显示混淆矩阵
cm = confusion_matrix(y_test, y_pred)
print("混淆矩阵:")
print(cm)
print(f"\nTP = {cm[1,1]}, FP = {cm[0,1]}, FN = {cm[1,0]}, TN = {cm[0,0]}")

# 可视化
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['负类(0)', '正类(1)'])
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('混淆矩阵可视化', fontsize=16)
plt.tight_layout()
plt.show()

---

## 2. 准确率（Accuracy）

### 2.1 公式

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN} = \frac{\text{正确预测的样本数}}{\text{总样本数}}$$

### 2.2 准确率的局限性

准确率在**类别不平衡**时会产生误导：

- 假设数据集中99%为负类，1%为正类
- 一个永远预测负类的「模型」准确率可达99%
- 但这个模型完全没有识别正类的能力！

> 💡 **NOAI竞赛提醒**：竞赛中数据常常不平衡，务必关注精确率、召回率等指标。

In [ ]:
from sklearn.metrics import accuracy_score

# 展示准确率在类别不平衡时的局限性
np.random.seed(42)

# 模拟极度不平衡数据：990个负类，10个正类
y_imbalanced = np.array([0]*990 + [1]*10)
np.random.shuffle(y_imbalanced)

# 笨模型：全部预测为0
y_dumb_pred = np.zeros_like(y_imbalanced)

print("=== 准确率的局限性演示 ===")
print(f"数据分布：正类 {sum(y_imbalanced==1)} 个，负类 {sum(y_imbalanced==0)} 个")
print(f"\n笨模型（全部预测为负类）的准确率: {accuracy_score(y_imbalanced, y_dumb_pred):.4f}")
print("准确率高达99%，但模型毫无价值！")

# 对比上一个实验中的准确率
acc = accuracy_score(y_test, y_pred)
print(f"\n之前逻辑回归模型的准确率: {acc:.4f}")

---

## 3. 精确率（Precision）与召回率（Recall）

### 3.1 公式

$$\text{Precision} = \frac{TP}{TP + FP}$$

> 在所有**被预测为正类**的样本中，有多少是真正正类的？

$$\text{Recall} = \frac{TP}{TP + FN}$$

> 在所有**实际为正类**的样本中，有多少被正确找出来了？

### 3.2 精确率与召回率的权衡

| 场景 | 侧重点 | 原因 |
|------|--------|------|
| 垃圾邮件检测 | **高精确率** | 把正常邮件误判为垃圾邮件代价高 |
| 癌症筛查 | **高召回率** | 漏诊一个癌症患者的代价远高于误报 |
| 搜索引擎 | **高精确率** | 用户希望前面的结果都相关 |
| 安检系统 | **高召回率** | 宁可误报，不可漏检危险品 |

> 🎯 **核心矛盾**：提高阈值 → 精确率上升、召回率下降；降低阈值 → 召回率上升、精确率下降。

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print("=== 精确率与召回率 ===")
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print(f"精确率 (Precision): {precision:.4f}")
print(f"召回率 (Recall):    {recall:.4f}")
print(f"准确率 (Accuracy):  {accuracy_score(y_test, y_pred):.4f}")

# 展示阈值变化对精确率/召回率的影响
y_proba = model.predict_proba(X_test)[:, 1]  # 正类的概率

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
print("\n=== 不同阈值下的精确率与召回率 ===")
print(f"{'阈值':^8} | {'精确率':^10} | {'召回率':^10} | {'F1分数':^10}")
print("-" * 48)

for thresh in thresholds:
    y_thresh_pred = (y_proba >= thresh).astype(int)
    p = precision_score(y_test, y_thresh_pred, zero_division=0)
    r = recall_score(y_test, y_thresh_pred, zero_division=0)
    f = f1_score(y_test, y_thresh_pred, zero_division=0)
    print(f"  {thresh:^6.1f} |  {p:^10.4f} |  {r:^10.4f} |  {f:^10.4f}")

---

## 4. F1分数（F1-Score）

### 4.1 公式

F1分数是精确率和召回率的**调和平均数**：

$$F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

### 4.2 为什么用调和平均而不是算术平均？

- 算术平均：$\frac{P+R}{2}$，一个高一个低时仍能得较高分
- **调和平均**：对极端值更敏感，只有P和R都高时F1才高

例如：P=1.0, R=0.01 → 算术平均=0.505，但F1≈0.02

### 4.3 $F_\beta$ 分数

当需要给精确率或召回率不同权重时：

$$F_\beta = (1 + \beta^2) \times \frac{P \times R}{\beta^2 \times P + R}$$

- $\beta = 2$：更重视召回率（召回率的权重是精确率的4倍）
- $\beta = 0.5$：更重视精确率

In [ ]:
from sklearn.metrics import fbeta_score

print("=== F1分数与F-beta分数 ===")
f1 = f1_score(y_test, y_pred)
print(f"F1分数 (beta=1):  {f1:.4f}")
print(f"F2分数 (beta=2, 偏重召回): {fbeta_score(y_test, y_pred, beta=2):.4f}")
print(f"F0.5分数 (beta=0.5, 偏重精确): {fbeta_score(y_test, y_pred, beta=0.5):.4f}")

# 对比调和平均 vs 算术平均
print("\n=== 调和平均 vs 算术平均 ===")
print(f"{'P':>8} {'R':>8} {'算术平均':>10} {'调和平均(F1)':>14}")
print("-" * 44)
for p_val, r_val in [(1.0, 0.01), (0.9, 0.5), (0.8, 0.8), (0.5, 0.9)]:
    arithmetic = (p_val + r_val) / 2
    harmonic = 2 * p_val * r_val / (p_val + r_val)
    print(f"{p_val:>8.2f} {r_val:>8.2f} {arithmetic:>10.4f} {harmonic:>14.4f}")

---

## 5. ROC曲线与AUC值

### 5.1 基本概念

**ROC（Receiver Operating Characteristic）曲线**展示的是：
- 横轴：**FPR（False Positive Rate，假正率）** = $\frac{FP}{FP + TN}$
- 纵轴：**TPR（True Positive Rate，真正率）** = $\frac{TP}{TP + FN}$ = 召回率

**AUC（Area Under Curve）**是ROC曲线下的面积：
- AUC = 1.0：完美分类器
- AUC = 0.5：随机猜测（等同于抛硬币）
- AUC < 0.5：比随机还差

### 5.2 ROC曲线的优势

- **对阈值不敏感**：综合考虑了所有可能的分类阈值
- **对类别不平衡具有鲁棒性**：因为使用的是率（比率）而非绝对数量

> ⚠️ **注意**：在极度不平衡数据上，ROC曲线可能过于乐观，此时应结合PR曲线使用。

In [ ]:
from sklearn.metrics import roc_curve, auc, roc_auc_score

# 计算ROC曲线数据
y_proba = model.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds_roc = roc_curve(y_test, y_proba)
roc_auc_value = auc(fpr, tpr)

print(f"AUC值: {roc_auc_value:.4f}")
print(f"(使用roc_auc_score: {roc_auc_score(y_test, y_proba):.4f})")

# 绘制ROC曲线
fig, ax = plt.subplots(figsize=(9, 7))

# 绘制ROC曲线
ax.plot(fpr, tpr, color='#E74C3C', linewidth=2.5,
        label=f'ROC曲线 (AUC = {roc_auc_value:.4f})')

# 绘制对角线（随机猜测）
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1.5,
        label='随机猜测 (AUC = 0.5)')

# 标注关键点
ax.scatter(fpr[np.argmin(np.abs(thresholds_roc - 0.5))],
           tpr[np.argmin(np.abs(thresholds_roc - 0.5))],
           color='#2C3E50', s=100, zorder=5, label='阈值=0.5')

ax.fill_between(fpr, tpr, alpha=0.1, color='#E74C3C')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('假正率 (FPR)', fontsize=14)
ax.set_ylabel('真正率 (TPR / Recall)', fontsize=14)
ax.set_title('ROC曲线 - Receiver Operating Characteristic', fontsize=16)
ax.legend(loc='lower right', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 6. PR曲线（Precision-Recall Curve）

### 6.1 基本概念

**PR曲线**展示的是精确率与召回率之间的关系：
- 横轴：召回率（Recall）
- 纵轴：精确率（Precision）

### 6.2 PR曲线 vs ROC曲线

| 特性 | ROC曲线 | PR曲线 |
|------|---------|--------|
| 横轴 | FPR | Recall |
| 纵轴 | TPR | Precision |
| 基线 | 对角线（AUC=0.5） | 正类比例的水平线 |
| 类别不平衡 | 鲁棒，但可能过于乐观 | 更敏感，更能反映真实表现 |
| 适用场景 | 各类场景通用 | 正类稀少时更推荐 |

> 💡 **经验法则**：当正类比例小于10%时，PR曲线比ROC曲线更能反映模型的真实性能。

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

# 计算PR曲线数据
precisions, recalls, thresholds_pr = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)

print(f"Average Precision (AP): {ap:.4f}")

# 绘制PR曲线
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左图：PR曲线
axes[0].plot(recalls, precisions, color='#2980B9', linewidth=2.5,
             label=f'PR曲线 (AP = {ap:.4f})')

# 基线 = 正类比例
baseline = sum(y_test == 1) / len(y_test)
axes[0].axhline(y=baseline, color='gray', linestyle='--', linewidth=1.5,
               label=f'基线 (正类比例 = {baseline:.4f})')

axes[0].fill_between(recalls, precisions, alpha=0.1, color='#2980B9')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('召回率 (Recall)', fontsize=13)
axes[0].set_ylabel('精确率 (Precision)', fontsize=13)
axes[0].set_title('PR曲线 - Precision-Recall Curve', fontsize=15)
axes[0].legend(loc='upper right', fontsize=11)
axes[0].grid(True, alpha=0.3)

# 右图：ROC曲线（对比）
axes[1].plot(fpr, tpr, color='#E74C3C', linewidth=2.5,
             label=f'ROC (AUC = {roc_auc_value:.4f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1.5,
             label='随机猜测')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#E74C3C')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('假正率 (FPR)', fontsize=13)
axes[1].set_ylabel('真正率 (TPR)', fontsize=13)
axes[1].set_title('ROC曲线 - 对比参考', fontsize=15)
axes[1].legend(loc='lower right', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 7. 多分类问题的评估

当类别数 > 2时，我们需要将二分类指标扩展到多分类场景。

### 7.1 三种平均策略

假设有3个类别，各类别的F1分数分别为 $F_1^{(1)}, F_1^{(2)}, F_1^{(3)}$：

#### Macro平均
$$\text{Macro-}F1 = \frac{1}{K} \sum_{k=1}^{K} F_1^{(k)}$$

- 每个类别的指标**等权平均**
- 对所有类别一视同仁
- 少数类也有相同的权重

#### Micro平均
$$\text{Micro-}F1 = \frac{\sum_{k} TP_k}{\sum_{k}(TP_k + FP_k)}$$

- 将所有类别的TP、FP加总后计算
- 等价于**全局准确率**（在多分类one-vs-rest场景下）
- 受大类影响更大

#### Weighted平均
$$\text{Weighted-}F1 = \sum_{k=1}^{K} w_k \cdot F_1^{(k)}$$

- 每个类别按其**样本数占比**加权
- 折中方案

> 🎯 **NOAI竞赛建议**：
>- 数据平衡时用 **Macro** 平均
>- 关注整体性能时用 **Weighted** 平均
>- 竞赛排名通常看 **Macro F1**

In [ ]:
from sklearn.metrics import (classification_report,
                              precision_recall_fscore_support)
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

# 使用鸢尾花数据集（3分类）
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_i, y_train_i)
y_pred_i = rf.predict(X_test_i)

# classification_report：一站式报告
print("=== 分类报告 ===")
print(classification_report(y_test_i, y_pred_i,
                            target_names=iris.target_names))

# 手动计算三种平均
p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
    y_test_i, y_pred_i, average='macro')
p_micro, r_micro, f1_micro, _ = precision_recall_fscore_support(
    y_test_i, y_pred_i, average='micro')
p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_test_i, y_pred_i, average='weighted')

print("\n=== 三种平均方式对比 ===")
print(f"{'方式':<12} {'Precision':>12} {'Recall':>12} {'F1-Score':>12}")
print("-" * 52)
print(f"{'Macro':<12} {p_macro:>12.4f} {r_macro:>12.4f} {f1_macro:>12.4f}")
print(f"{'Micro':<12} {p_micro:>12.4f} {r_micro:>12.4f} {f1_micro:>12.4f}")
print(f"{'Weighted':<12} {p_weighted:>12.4f} {r_weighted:>12.4f} {f1_weighted:>12.4f}")

# 多分类混淆矩阵
cm_multi = confusion_matrix(y_test_i, y_pred_i)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_multi,
                              display_labels=iris.target_names)
disp.plot(ax=ax, cmap='Greens', values_format='d')
plt.title('多分类混淆矩阵（鸢尾花）', fontsize=16)
plt.tight_layout()
plt.show()

---

## 8. 综合实战：使用sklearn的评估工具

下面我们通过一个完整的案例，展示如何系统地评估一个分类模型。

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# 重新生成一个更有挑战性的不平衡数据集
X_hard, y_hard = make_classification(
    n_samples=2000, n_features=20, n_informative=10,
    n_classes=2, weights=[0.85, 0.15], flip_y=0.05,
    random_state=42)

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_hard, y_hard, test_size=0.3, random_state=42, stratify=y_hard)

# 构建Pipeline
pipe = make_pipeline(StandardScaler(), SVC(kernel='rbf', probability=True, random_state=42))
pipe.fit(X_train_h, y_train_h)
y_pred_h = pipe.predict(X_test_h)
y_proba_h = pipe.predict_proba(X_test_h)[:, 1]

# ====== 综合评估报告 ======
print("=" * 60)
print("         模型综合评估报告")
print("=" * 60)

print(f"\n📊 数据分布:")
print(f"  训练集: 正类 {sum(y_train_h==1)} ({sum(y_train_h==1)/len(y_train_h)*100:.1f}%), "
      f"负类 {sum(y_train_h==0)} ({sum(y_train_h==0)/len(y_train_h)*100:.1f}%)")
print(f"  测试集: 正类 {sum(y_test_h==1)} ({sum(y_test_h==1)/len(y_test_h)*100:.1f}%), "
      f"负类 {sum(y_test_h==0)} ({sum(y_test_h==0)/len(y_test_h)*100:.1f}%)")

print(f"\n📈 核心指标:")
print(f"  Accuracy:  {accuracy_score(y_test_h, y_pred_h):.4f}")
print(f"  Precision: {precision_score(y_test_h, y_pred_h):.4f}")
print(f"  Recall:    {recall_score(y_test_h, y_pred_h):.4f}")
print(f"  F1-Score:  {f1_score(y_test_h, y_pred_h):.4f}")
print(f"  AUC-ROC:   {roc_auc_score(y_test_h, y_proba_h):.4f}")
print(f"  AP (PR):   {average_precision_score(y_test_h, y_proba_h):.4f}")

# 混淆矩阵
cm_h = confusion_matrix(y_test_h, y_pred_h)
print(f"\n📋 混淆矩阵:")
print(f"           预测负  预测正")
print(f"  实际负    {cm_h[0,0]:>5d}   {cm_h[0,1]:>5d}")
print(f"  实际正    {cm_h[1,0]:>5d}   {cm_h[1,1]:>5d}")

# 绘制综合图
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 混淆矩阵
ConfusionMatrixDisplay(confusion_matrix=cm_h,
                        display_labels=['负类', '正类']).plot(
    ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('混淆矩阵', fontsize=14)

# ROC曲线
fpr_h, tpr_h, _ = roc_curve(y_test_h, y_proba_h)
axes[1].plot(fpr_h, tpr_h, color='#E74C3C', linewidth=2,
              label=f'ROC (AUC={auc(fpr_h, tpr_h):.3f})')
axes[1].plot([0,1], [0,1], 'k--', alpha=0.5)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC曲线', fontsize=14)
axes[1].legend(); axes[1].grid(alpha=0.3)

# PR曲线
prec_h, rec_h, _ = precision_recall_curve(y_test_h, y_proba_h)
axes[2].plot(rec_h, prec_h, color='#2980B9', linewidth=2,
              label=f'PR (AP={average_precision_score(y_test_h, y_proba_h):.3f})')
baseline_h = sum(y_test_h == 1) / len(y_test_h)
axes[2].axhline(y=baseline_h, color='gray', linestyle='--', alpha=0.5,
               label=f'基线={baseline_h:.3f}')
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
axes[2].set_title('PR曲线', fontsize=14)
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## 9. 练习题

### 📝 练习1：手动计算指标

给定以下混淆矩阵：

| | 预测正 | 预测负 |
|---|---|---|
| 实际正 | 80 | 20 |
| 实际负 | 10 | 90 |

请手动计算：
- (a) 准确率
- (b) 精确率
- (c) 召回率
- (d) F1分数

In [ ]:
# 练习1：手动计算
TP, FP, FN, TN = 80, 10, 20, 90

# 请在下方填写你的计算
accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)

print(f"(a) 准确率:   {accuracy:.4f}")
print(f"(b) 精确率:   {precision:.4f}")
print(f"(c) 召回率:   {recall:.4f}")
print(f"(d) F1分数:   {f1:.4f}")

# 验证
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
y_true_ex = [1]*80 + [1]*20 + [0]*10 + [0]*90
y_pred_ex = [1]*80 + [0]*20 + [1]*10 + [0]*90

print(f"\n--- sklearn验证 ---")
print(f"Accuracy:  {accuracy_score(y_true_ex, y_pred_ex):.4f}")
print(f"Precision: {precision_score(y_true_ex, y_pred_ex):.4f}")
print(f"Recall:    {recall_score(y_true_ex, y_pred_ex):.4f}")
print(f"F1:        {f1_score(y_true_ex, y_pred_ex):.4f}")

### 📝 练习2：不同模型对比

请在同一个数据集上训练3个不同的分类模型，对比它们的ROC曲线和AUC值。

In [ ]:
# 练习2：模型对比
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier

# 使用之前的二分类数据
models = {
    '逻辑回归': LogisticRegression(max_iter=1000, random_state=42),
    '决策树': DecisionTreeClassifier(max_depth=5, random_state=42),
    '梯度提升': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name, mdl in models.items():
    mdl.fit(X_train, y_train)
    y_prob = mdl.predict_proba(X_test)[:, 1]

    # ROC
    fpr_m, tpr_m, _ = roc_curve(y_test, y_prob)
    auc_m = auc(fpr_m, tpr_m)
    axes[0].plot(fpr_m, tpr_m, linewidth=2, label=f'{name} (AUC={auc_m:.3f})')

    # PR
    prec_m, rec_m, _ = precision_recall_curve(y_test, y_prob)
    ap_m = average_precision_score(y_test, y_prob)
    axes[1].plot(rec_m, prec_m, linewidth=2, label=f'{name} (AP={ap_m:.3f})')

# ROC图
axes[0].plot([0,1], [0,1], 'k--', alpha=0.3)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC曲线对比', fontsize=14)
axes[0].legend(fontsize=11); axes[0].grid(alpha=0.3)

# PR图
axes[1].axhline(y=sum(y_test==1)/len(y_test), color='gray', linestyle='--', alpha=0.3)
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('PR曲线对比', fontsize=14)
axes[1].legend(fontsize=11); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== 各模型指标汇总 ===")
print(f"{'模型':<12} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'AUC':>10}")
print("-" * 66)
for name, mdl in models.items():
    y_p = mdl.predict(X_test)
    y_prob = mdl.predict_proba(X_test)[:, 1]
    print(f"{name:<12} {accuracy_score(y_test, y_p):>10.4f} "
          f"{precision_score(y_test, y_p):>10.4f} "
          f"{recall_score(y_test, y_p):>10.4f} "
          f"{f1_score(y_test, y_p):>10.4f} "
          f"{roc_auc_score(y_test, y_prob):>10.4f}")

### 📝 练习3（挑战）：自定义评估场景

假设你在做一个**疾病诊断**模型：
- 疾病发病率仅1%
- 你更关注不漏诊（高召回率）

请完成以下任务：
1. 创建极度不平衡数据（正类1%）
2. 训练模型并调整决策阈值
3. 找到召回率大于等于0.9时的最佳精确率
4. 绘制该场景下的PR曲线

In [ ]:
# 练习3：疾病诊断场景（参考答案框架）
X_disease, y_disease = make_classification(
    n_samples=5000, n_features=15, n_informative=8,
    weights=[0.99, 0.01], flip_y=0.02, random_state=42)

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_disease, y_disease, test_size=0.3, random_state=42, stratify=y_disease)

print(f"训练集正类比例: {sum(y_train_d==1)/len(y_train_d)*100:.2f}%")
print(f"测试集正类比例: {sum(y_test_d==1)/len(y_test_d)*100:.2f}%")

# 训练
pipe_d = make_pipeline(StandardScaler(),
                       SVC(kernel='rbf', probability=True, random_state=42))
pipe_d.fit(X_train_d, y_train_d)
y_proba_d = pipe_d.predict_proba(X_test_d)[:, 1]

# 找到召回率>=0.9的最佳阈值
prec_d, rec_d, thr_d = precision_recall_curve(y_test_d, y_proba_d)

# 在召回率>=0.9中找精确率最高的点
mask = rec_d >= 0.9
if mask.any():
    best_idx = np.where(mask)[0][0]
    best_thr = thr_d[min(best_idx, len(thr_d)-1)] if best_idx < len(thr_d) else 0.0
    print(f"\n召回率>=0.9时：")
    print(f"  最佳精确率: {prec_d[best_idx]:.4f}")
    print(f"  对应召回率: {rec_d[best_idx]:.4f}")
    print(f"  建议阈值:   {best_thr:.4f}")

# 绘图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PR曲线
axes[0].plot(rec_d, prec_d, 'b-', linewidth=2,
             label=f'PR (AP={average_precision_score(y_test_d, y_proba_d):.3f})')
axes[0].axhline(y=0.01, color='gray', linestyle='--', alpha=0.5, label='基线')
if mask.any():
    axes[0].scatter([rec_d[best_idx]], [prec_d[best_idx]], c='red', s=100, zorder=5,
                   label=f'最优点(P={prec_d[best_idx]:.3f}, R={rec_d[best_idx]:.3f})')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('疾病诊断 PR曲线'); axes[0].legend(); axes[0].grid(alpha=0.3)

# ROC曲线
fpr_d, tpr_d, _ = roc_curve(y_test_d, y_proba_d)
axes[1].plot(fpr_d, tpr_d, 'r-', linewidth=2,
             label=f'ROC (AUC={auc(fpr_d, tpr_d):.3f})')
axes[1].plot([0,1], [0,1], 'k--', alpha=0.3)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('疾病诊断 ROC曲线'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## 📌 本节总结

| 指标 | 公式 | 适用场景 |
|------|------|----------|
| Accuracy | $\frac{TP+TN}{TP+TN+FP+FN}$ | 类别平衡时 |
| Precision | $\frac{TP}{TP+FP}$ | 误报代价高 |
| Recall | $\frac{TP}{TP+FN}$ | 漏报代价高 |
| F1 | $\frac{2PR}{P+R}$ | 需要平衡P和R |
| AUC-ROC | ROC曲线下面积 | 综合评估分类能力 |
| AP | PR曲线下面积 | 不平衡数据 |

**关键要点：**
1. 没有任何单一指标能全面评估模型，需要结合多个指标
2. 类别不平衡时，准确率具有欺骗性
3. 精确率与召回率存在权衡关系
4. ROC曲线对阈值不敏感，适合整体评估
5. PR曲线在正类稀少时更可靠
6. 多分类时注意选择合适的平均策略